# Explore a pulse sequence — drag a box to zoom

The interactive counterpart to `visualize_sequence.ipynb` (which is static/inline).

| gesture | action |
|---|---|
| **drag a box** | zoom to that time window; if the box is tall, also to that lane range |
| double-click | reset to the full sequence |
| `r` | reset (click the canvas once first so it has keyboard focus) |
| scroll | zoom the time axis about the cursor |
| shift + drag | pan |

Zooming **re-renders** rather than just rescaling: label density, the ns/µs unit
and the envelope overlay are all chosen for the window in view, and marks outside
it are culled. So zooming into a single pulse shows its actual ramp shape.

Sections below: pick an example folder → drive the viewport from code → step
sweep points → inspect **control flow** (loops, `test`, `repeat_until`).

The same `SequenceView` class is what `qt_widget.py` wraps for acadia_gui, so
anything that works here works there.

In [ ]:
%matplotlib widget

import logging
import sys
from pathlib import Path



here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "sequence_viz" / "__init__.py").is_file():
        sys.path.insert(0, str(candidate))
        break

for name in ("acadia", "qmsmt_runtime_loader"):
    logging.getLogger(name).setLevel(logging.ERROR)

import sequence_viz as sv

In [ ]:
# A spread of real runs, from a 1-block Rabi to a 24-block sequence with a
# repeat_until reset loop. See docs/EXAMPLE_FOLDERS.md for the full list and what
# each one exercises.
FT = "/path/to/data_root"
EXAMPLES = {
    # name                       folder
    "rabi (1 block)":            f"{FT}/FT_RA_Calibration/qubit_cal/qb_rabi/260727_010800",
    "T2 echo (register delay)":  f"{FT}/FT_RA_Calibration/qubit_cal/qb_t2e/260727_010933",
    "chevron (4096 points)":     f"{FT}/FT_RA_Calibration/qubit_cal/qb_R1_chevron/260727_011131",
    "1 qubit tomo (test)":       f"{FT}/LB_Qubit_Char/1_Qubit_Tomo/260723/164303",
    "AllXY (42 blocks, test)":   f"{FT}/LB_Qubit_Char/AllXY_prelim_test/260723/165848",
    "coherence (repeat_until)":  f"{FT}/FT_RA_Calibration/qubit_cal/R1_coherence/260727_011426",
    "BS amp sweep + DR tomo":    f"{FT}/RA_BeamSplitting/RA_BeamSplitting/QM_QM_BS_AmpSweep_DR_Tomo/RA_R1_Bus/260726/005805",
    "blob cal (17 blocks)":      f"{FT}/FT_RB_Calibration/qubit_cal/bs_blob_cal/20260710_134841/swap_swapRep064",
}

DATA_FOLDER = EXAMPLES["rabi (1 block)"]

view = sv.explore_folder(
    DATA_FOLDER,
    point=0,                     # which acadia.run() to show
    resolve_indeterminate=2,     # cycles to assume for unresolvable register lengths
    use_saved_qmsmt=True,        # the folder's own acadia_qmsmt.py; False = installed
    envelope_mode="magnitude",   # or "iq"
    envelope_scale="per-pulse",  # "channel" / "shared" / "absolute" to compare amplitudes
)
print(view.trace.summary())
view.ax.figure

## Driving the viewport from code

Useful when you know which block you care about — the figure above updates in place.

Note the two lists: `trace.blocks` is what was **compiled** (one entry per
`channel_synchronizer`), `trace.placements` is what **executes** — loops unrolled, and
untaken `test` bodies dropped. Placements are what the plot draws.

In [ ]:
trace = view.trace
ns = trace.ns_per_cycle

print(f"{len(trace.blocks)} compiled blocks -> {len(trace.placements)} executed placements")
for plc in trace.placements:
    tag = f" (pass {plc.iteration + 1})" if getattr(plc, "iteration", 0) else ""
    print(f"  block {plc.index:3d}{tag:11s} {plc.start * ns:9.0f} -> {plc.stop * ns:9.0f} ns  "
          f"blocking={plc.blocking}  {len(plc.commands)} commands")

In [ ]:
PLACEMENT = -1                   # index into trace.placements

plc = trace.placements[PLACEMENT]
pad = max(plc.length * ns * 0.1, 50)
view.set_window(plc.start * ns - pad, plc.stop * ns + pad)
print(f"showing block {plc.index}: {view.xlim_ns[0]:.0f} -> {view.xlim_ns[1]:.0f} ns")

In [ ]:
view.reset()

## Stepping through sweep points

One dry run captured **every** point — the schedule is compiled once and shared, so
only the pulse data and the cache differ. Switching is instant, no re-tracing
(`iterations` is forced to 1 during the dry run, since iterations only repeat the
same points).

For a register-driven sweep (T1, T2, chevron) the delay is read out of the cache
per point, so the timeline length changes as you step.

In [ ]:
print(f"{trace.n_points} points captured "
      f"(iterations forced to 1: {trace.iterations_forced})")
if trace.register_sources:
    print("register -> cache word:", trace.register_sources)

for p in range(min(6, trace.n_points)):
    view.set_point(p)                     # instant; the figure above updates
    regs = {c.symbolic: round(c.length * ns) for c in trace.commands if c.symbolic}
    print(f"  point {p:3d}: total {trace.length_ns:9.0f} ns"
          + (f"  registers(ns)={regs}" if regs else ""))

view.set_point(0)

## Control flow — what is drawn, and what is only assumed

Every region under a `loop`, `test` or `repeat_until` gets a dashed outline and a caption
that says exactly how much is known:

| context | what the plot shows | validated? |
|---|---|---|
| `loop(N)` | all N passes, unrolled, each labelled `pass k of N` | ✅ ≤0.09 ns on hardware |
| `test(cond)`, `speculation=True` (default) | the arm that runs — decided from the cache if possible, else assumed and said so | ✅ 0.04 ns taken / 0.10 ns skipped |
| `test(cond)`, `speculation=False` | body drawn, captioned **TIMING NOT MODELLED** | ❌ KI_004 — body is out of line |
| `repeat_until(cond)` | **one** pass, captioned as data-dependent | ➖ trip count is unknowable off-hardware |

So a `repeat_until` reset loop makes the real sequence *longer* than the drawing, never
shorter. `docs/VALIDATION.md` has the hardware numbers behind that table.

In [ ]:
regions = sv.branch_regions(trace)
print(f"{len(regions)} control-flow regions "
      f"({len(trace.assumed_paths)} assumed, {len(trace.unsupported_paths)} unsupported)\n")
for start, stop, context, info in regions:
    print(f"  {start * ns:8.0f} -> {stop * ns:8.0f} ns   "
          f"{sv.branch_caption(trace, context, info)}")
    if len(context) > 1:                       # nested: outer contexts first
        for outer in context[:-1]:
            print(f"{'':30s}inside {outer['kind']}({outer['condition']})")

### Forcing the other arm, or another loop count

Neither needs a re-trace — the compiled schedule does not depend on which path you choose
to *draw*. Set the override, re-lay out, and the figure above updates.

In [ ]:
# block index -> take the test body? (True/False); and block index -> passes to draw
CHOICES = {}                      # e.g. {12: False} to see the skipped arm
COUNTS = {}                       # e.g. {7: 3} to draw 3 passes of a repeat_until

trace.path_choices.update(CHOICES)
trace.loop_counts.update(COUNTS)
view.relayout()
print(f"{len(trace.placements)} placements, total {trace.length_ns:.0f} ns")

## A live runtime instead of a folder

In [ ]:
# rt = SomeRuntime(**config)
# view = sv.explore_runtime(rt)
# view.ax.figure